# Flood summaries for ThinkHazard

This script performs flood hazard ranking by administrative unit using global-extent
Fathom tiles hosted on AWS S3, rather than country-extent locally downloaded data.

The hazard ranking is based on:
- Value threshold: Minimum flood depth (cm) to consider
- Area threshold: Minimum percentage of area affected
- Hazard score: Count of return periods meeting both thresholds (0-3)


In [ ]:
import os, time, io, json, sys
import urllib3
import boto3

import geopandas as gpd
import pandas as pd
import numpy as np

from functools import reduce
from urllib3.exceptions import InsecureRequestWarning
from botocore import UNSIGNED
from botocore.config import Config
from tqdm.notebook import tqdm

# Import helper functions
from gfdrr_helper import *

urllib3.disable_warnings(InsecureRequestWarning)

def tPrint(s):
    """prints the time along with the message"""
    print("%s\t%s" % (time.strftime("%H:%M:%S"), s))

s3_client = boto3.client('s3', verify=False, config=Config(signature_version=UNSIGNED))

%load_ext autoreload
%autoreload 2

In [ ]:
local_folder = "C:/WBG/Work/Projects/ThinkHazard"
out_folder = os.path.join(local_folder, "FATHOM_summaries")
map_folder = os.path.join(local_folder, "FATHOM_maps")
for tF in [out_folder, map_folder]:
    if not os.path.exists(tF):
        os.makedirs(tF)
vrt_folder = r"C:\WBG\Work\data\FATHOM"
s3_bucket = "wbg-geography01"
s3_prefix = "FATHOM"
return_periods = [10, 100, 500, 1000]
flood_files = [
    ["FU", "FLOOD_MAP-1ARCSEC-NW_OFFSET-1in{rp}-FLUVIAL-UNDEFENDED-DEPTH-2020-PERCENTILE50-v3.1.vrt"],
    ["CU", "FLOOD_MAP-1ARCSEC-NW_OFFSET-1in{rp}-COASTAL-UNDEFENDED-DEPTH-2020-PERCENTILE50-v3.1.vrt"],
    ['PD', "FLOOD_MAP-1ARCSEC-NW_OFFSET-1in{rp}-PLUVIAL-DEFENDED-DEPTH-2020-PERCENTILE50-v3.1.vrt"]
]

admin_boundaries_file = r"C:\WBG\Work\data\ADMIN\NEW_WB_BOUNDS\FOR_PUBLICATION\crs_4326\parquet\WB_GAD_ADM2.parquet"
inA = gpd.read_parquet(admin_boundaries_file)

In [ ]:
cur_out_folder = os.path.join(out_folder, "FATHOM_Detailed")
if not os.path.exists(cur_out_folder):
    os.makedirs(cur_out_folder)

cur_map_folder = os.path.join(map_folder, "FATHOM_Detailed")
if not os.path.exists(cur_map_folder):
    os.makedirs(cur_map_folder)

with rasterio.Env(GDAL_HTTP_UNSAFESSL='YES'):
    for sel_country in inA['ISO_A3'].unique():
        all_res = []
        out_file = os.path.join(cur_out_folder, f"FATHOM_ThinkHazard_summary_{sel_country}.csv")
        sel_a = inA[inA['ISO_A3'] == sel_country]                           
        if not os.path.exists(out_file) and not (sel_country in ["FJI",'RUS']):
            tPrint(f"Processing country: {sel_country}")
            for lbl, raster_file in flood_files:
                for return_period in return_periods:
                    tPrint(f"Processing {lbl} for {return_period} year return period")
                    sel_raster_file = raster_file.format(rp=return_period)
                    sel_raster = f"s3://{s3_bucket}/{s3_prefix}/{sel_raster_file}"
                    res_a = calculate_think_hazard_score(sel_a, sel_raster, 
                                                         depth_threshold=50, idx_col='ADM2CD_c',
                                                         all_touched=True, min_val=0, max_val=10000, no_data=-32768)
                    res_a.rename(columns={'frac_area_flooded': f'frac_area_flooded_{lbl}_{return_period}yr',
                                          #'area_flooded': f'area_flooded_{lbl}_{return_period}yr',
                                          #'total_area': f'total_area_{lbl}_{return_period}yr'                                          
                                          }, inplace=True)
                    all_res.append(res_a)                    
            all_res_df = reduce(lambda left, right: pd.merge(left, right, on='ADM2CD_c', how='outer'), all_res) 
            all_res_df.to_csv(out_file, index=False)

            sel_a = inA[inA['ISO_A3'] == sel_country]                                   
            map_adm = pd.merge(sel_a, all_res_df, on='ADM2CD_c', how='left')
            map_flood(map_adm, return_period=100, out_file=os.path.join(cur_map_folder, f"flood_map_{sel_country}_100yr.png"))
        else:
            tPrint(f"File already exists for {sel_country}, skipping...")

# Extracting sample data

In [ ]:
sys.path.insert(0, "C:\\WBG\\Work\\Code\\GOSTrocks\\src")
import GOSTrocks.rasterMisc as rMisc
rMisc.clipRaster?

In [ ]:
temp_out_folder = "C:/Temp/FATHOM_TUN"
if not os.path.exists(temp_out_folder):
    os.makedirs(temp_out_folder)

sel_admin = inA.loc[inA['ISO_A3'] == "TUN"]
sel_admin.to_file(os.path.join(temp_out_folder, "TUN_admin.gpkg"), driver="GPKG")

return_periods = [10, 100, 500, 1000]
flood_files = [
    ["FU", "FLOOD_MAP-1ARCSEC-NW_OFFSET-1in{rp}-FLUVIAL-UNDEFENDED-DEPTH-2020-PERCENTILE50-v3.1.vrt"],
    ["CU", "FLOOD_MAP-1ARCSEC-NW_OFFSET-1in{rp}-COASTAL-UNDEFENDED-DEPTH-2020-PERCENTILE50-v3.1.vrt"],
    ['PD', "FLOOD_MAP-1ARCSEC-NW_OFFSET-1in{rp}-PLUVIAL-DEFENDED-DEPTH-2020-PERCENTILE50-v3.1.vrt"]
]

for return_period in return_periods:
    for lbl, raster_file in flood_files:
        temp_out_file = os.path.join(temp_out_folder, f"TUN_{lbl}_{return_period}yr.tif")
        if not os.path.exists(temp_out_file):
            sel_raster_file = raster_file.format(rp=return_period)
            sel_raster = f"s3://{s3_bucket}/{s3_prefix}/{sel_raster_file}"
            with rasterio.Env(GDAL_HTTP_UNSAFESSL='YES'):
                inR = rasterio.open(sel_raster)
                rMisc.clipRaster(inR, sel_admin, temp_out_file, crop=False)